# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [2]:
df = pd.read_csv('/content/clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [5]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [7]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [8]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
1,00114d74e963e47177db89bc70108537,-0.003994,-0.000001
2,001cd16732dc7d5bdf46b0d49996c271,0.047697,2.177779
3,001dc71143f45cb58aaccc2e94823c5a,-0.003778,0.177779
4,002b3009d069858b471918402fb237b7,-0.005795,0.000000


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

## Feature Engineering: Price Sensitivity

**Hypothesis:** The difference in off-peak prices between December and the preceding January might affect churn.

We will calculate this by:
1. Grouping price data by `id` and `price_date`.
2. Pivoting the data to have months as columns.
3. Calculating the difference: `December Price` - `January Price`.

In [9]:
import pandas as pd
import numpy as np

# Load data (ensuring dates are parsed)
df = pd.read_csv('clean_data_after_eda.csv')
price_df = pd.read_csv('price_data.csv')

# Ensure datetime format
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')

# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({
    'price_off_peak_var': 'mean',
    'price_off_peak_fix': 'mean'
}).reset_index()

# Pivot the table to make months columns
pivot_prices = monthly_price_by_id.pivot(index='id', columns='price_date', values=['price_off_peak_var', 'price_off_peak_fix'])

# Calculate the difference (Dec - Jan)
# Note: We access columns by their specific dates
diff_data = pd.DataFrame()
diff_data['id'] = pivot_prices.index
diff_data['diff_dec_jan_off_peak_var'] = pivot_prices['price_off_peak_var']['2015-12-01'].values - pivot_prices['price_off_peak_var']['2015-01-01'].values
diff_data['diff_dec_jan_off_peak_fix'] = pivot_prices['price_off_peak_fix']['2015-12-01'].values - pivot_prices['price_off_peak_fix']['2015-01-01'].values

# Merge this new feature into the main dataframe
df = pd.merge(df, diff_data, on='id', how='left')

print(f"Added price difference features. New shape: {df.shape}")
df[['id', 'diff_dec_jan_off_peak_var', 'diff_dec_jan_off_peak_fix']].head()

Added price difference features. New shape: (14606, 46)


,id,diff_dec_jan_off_peak_var,diff_dec_jan_off_peak_fix
0,24011ae4ebbe3035111d65fa7c15bc57,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,-0.006192,0.162916


## Creative Feature Engineering

To further improve the model, we will create additional features based on:
1.  **Contract Tenure:** How long has the client been with us? (Long-term clients might behave differently).
2.  **Seasonality:** Which month does the contract renew? (People might churn more in winter).
3.  **Log Transformation:** The consumption data is highly skewed. We will apply a log transformation to normalize it.

In [10]:
# Convert main dataframe dates to datetime objects
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# 1. Contract Tenure (in years)
# How long has the client been active?
df['tenure_years'] = (df['date_end'] - df['date_activ']).dt.days / 365

# 2. Seasonality (Month of Renewal)
# We extract the month to see if renewals in specific months lead to higher churn
df['renewal_month'] = df['date_renewal'].dt.month

# 3. Log Transformation of Consumption
# We add +1 to avoid log(0) errors
df['log_cons_12m'] = np.log10(df['cons_12m'] + 1)

print("Creative features added successfully.")
df[['id', 'tenure_years', 'renewal_month', 'log_cons_12m']].head()

Creative features added successfully.


,id,tenure_years,renewal_month,log_cons_12m
0,24011ae4ebbe3035111d65fa7c15bc57,3.002740,6,0.000000
1,d29c2c54acc38ff3c0614d0a653813dd,7.030137,8,3.668479
2,764c75f661154dac3a6c254cd082ea7d,6.005479,4,2.736397
3,bba03439a292a1e166f80264c16191cb,6.005479,3,3.200029
4,149d57cf92fc41cf94415803a877cb4b,6.150685,3,3.646011


## Final Data Preparation

Machine Learning models generally cannot handle raw date objects or identifier strings.
Now that we have extracted the useful information (tenure, months) from the dates, we can drop the original date columns.

In [11]:
# Identify columns to drop (dates and original high-skew columns if desired)
# We keep 'id' for now to identify predictions later, but we will remove dates
cols_to_drop = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']

model_data = df.drop(columns=cols_to_drop)

# Final check of the dataset
print("Final Dataset Shape:", model_data.shape)
print("Data Types:")
print(model_data.dtypes.value_counts())

# Display first rows
model_data.head()

Final Dataset Shape: (14606, 45)
Data Types:
float64    33
int64       7
object      4
int32       1
Name: count, dtype: int64


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,...,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,diff_dec_jan_off_peak_var,diff_dec_jan_off_peak_fix,tenure_years,renewal_month,log_cons_12m
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,0.00,0,0.0,1.78,0.114481,...,44.235794,2.086425,9.953056e+01,4.423670e+01,1,0.020057,3.700961,3.002740,6,0.000000
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,189.95,0,0.0,16.27,0.145711,...,0.000000,0.009485,1.217891e-03,0.000000e+00,0,-0.003767,0.177779,7.030137,8,3.668479
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,47.96,0,0.0,38.72,0.165794,...,0.000000,0.000004,9.450150e-08,0.000000e+00,0,-0.004670,0.177779,6.005479,4,2.736397
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,240.04,0,0.0,19.83,0.146694,...,0.000000,0.000003,0.000000e+00,0.000000e+00,0,-0.004547,0.177779,6.005479,3,3.200029
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,445.75,526,0.0,131.73,0.116900,...,0.000000,0.000011,2.896760e-06,4.860000e-10,0,-0.006192,0.162916,6.150685,3,3.646011
